# 03 · Predicting 6-month customer value

**Question:** given a customer's behaviour up to 9 June 2011, how much will they spend in the next 6 months (to 8 Dec 2011, which includes the Q4 peak)?

| Model | Idea |
|---|---|
| Naive | next 6 months = last 6 months |
| **BG/NBD + Gamma-Gamma** | probabilistic *buy-till-you-die* model (4 + 3 parameters), [implemented from the papers](../src/retail_clv/btyd.py) |
| LightGBM (behavioural) | gradient boosting on 17 behavioural + calendar features, Tweedie loss |
| LightGBM + BTYD features | the same, plus BG/NBD outputs as features (hybrid) |

**No leakage, by design:** features only see `[cutoff − 180 d, cutoff)`. LightGBM is trained on 7 stacked monthly cutoffs (Jun–Dec 2010) whose target windows all end on or before the test cutoff. BTYD features are refitted at every cutoff using only prior history. A unit test (`tests/test_clv.py::test_features_ignore_future_transactions`) enforces this.

In [1]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from retail_clv import data, plots

plots.apply_style()
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 20)
tx, report = data.clean(data.load_raw(data.download()))
inv = data.invoices(tx)
metrics = json.loads(Path("reports/metrics.json").read_text())

In [2]:
c = metrics["clv"]
print("training cutoffs:", ", ".join(c["train"]["cutoffs"]), f"-> {c['train']['rows']:,} rows")
print(
    f"test population: {c['test']['customers']:,} customers, "
    f"{c['test']['share_with_future_purchase']:.0%} purchase again, actual revenue £{c['test']['actual_revenue']:,.0f}"
)
res = pd.read_csv("reports/clv_test_metrics.csv", index_col=0)
res

training cutoffs: 2010-06-09, 2010-07-09, 2010-08-09, 2010-09-09, 2010-10-09, 2010-11-09, 2010-12-09 -> 20,737 rows
test population: 2,625 customers, 70% purchase again, actual revenue £3,763,107


,mae,rmse,spearman,top20_capture,top20_capture_vs_oracle,total_bias
Naive (last 6 months),895.520,"4,109.778",0.600,0.695,0.884,-0.170
BG/NBD + Gamma-Gamma,805.385,"2,949.943",0.645,0.710,0.903,-0.079
LightGBM (behavioural),865.683,"3,978.983",0.601,0.698,0.887,-0.189
LightGBM + BTYD features,839.318,"3,787.884",0.645,0.711,0.905,-0.124


* `top20_capture` is the share of actual future revenue generated by the 20 % of customers each model ranks highest. This is what a targeting campaign cares about.
* `top20_capture_vs_oracle` compares that with a model that knew the future.
* `total_bias` is the error in the forecast of total revenue (negative means under-forecast).

![capture](../reports/figures/clv_capture_curves.png)

## What the comparison shows

1. **The probabilistic BG/NBD + Gamma-Gamma model is the best single model.** It has the lowest MAE and RMSE, ties for the best rank correlation, and has by far the smallest bias in total revenue (−8 % vs −12 % to −19 %). It does this with 7 parameters and no labelled training data.
2. **Gradient boosting on behavioural features barely beats the naive rule.** Past spend already explains most of the ranking. Adding BTYD outputs as features lets LightGBM match BG/NBD on ranking, but not on calibration of totals.
3. **All models capture about 90 % of what perfect foresight would** in the top 20 %. The rest is largely unpredictable from transaction history alone (new needs, one-off large orders).

**Recommendation:** deploy BG/NBD + Gamma-Gamma as the production CLV model. It is simpler, explainable, needs no retraining labels, and also provides P(alive) for churn targeting. Keep the hybrid LightGBM as a challenger and retest it once richer features (product mix, web behaviour, marketing touches) are available.

## Calibration by decile

![deciles](../reports/figures/clv_deciles.png)

Both models rank customers well (predicted and actual values rise together). Both **under-predict the top decile**: a handful of wholesalers place very large Q4 orders that no model anticipates from history.

## Model parameters & drivers

In [3]:
p = metrics["clv"]["btyd_params_holdout"]
for name, params in p.items():
    print(f"{name:12s}", "  ".join(f"{k}={v:.3f}" for k, v in params.items()))

bgnbd        r=0.260  alpha=8.707  a=0.123  b=1.360
gamma_gamma  p=11.783  q=0.891  v=11.715


* BG/NBD: `r/alpha` ≈ 0.03 purchases per day for the average active customer (about one order a month), and the small `a` relative to `b` means a low drop-out probability after each purchase. Churn is slow but real.
* Gamma-Gamma: `q < 1` means spend per order is so heavy-tailed (wholesalers) that the theoretical population mean is infinite. The implementation detects this and falls back to the empirical mean for customers without repeat purchases. This is a real-data edge case that off-the-shelf tutorials rarely mention.

![drivers](../reports/figures/clv_feature_importance.png)

## Limitations

* Single retailer and a two-year history. The test period is one season, so the results carry seasonal noise.
* The target is net revenue, not margin. Profit-based CLV would need product costs.
* New customers who first appear after the cutoff are outside the scope of any of these models.